This gets all the software we'll need into your working environment.

In [ ]:
import json
import pyodide.http
import zipfile
from math import sqrt

from tqdm import tqdm
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pyarrow as pa
import shapely

import data

# 1. Introduction

_(Jim gives a presentation.)_

# 2. Gerrymandering challenge

You've been appointed by the Supreme Chancellor to draw the boundaries that define the galaxy's 9 sectors. Each planet in a sector gets one vote to elect a senator for its sector: 9 senators total. The Chancellor's Purple Party is popular in 200 planets; the opposing party is popular in 250 planets. Even though the Purple Party is less popular overall, draw the boundaries in a way to ensure a majority, at least 5 of the 9 senators, for the Purple Party.

I've managed to get as many as 7 Purple Party senators. Is it possible to get 8?

<a href="/2025-07-12-scipy-teen-track/gerrymandering-galaxy.svg"><img src="img/gerrymandering-galaxy.svg" width="500"></a>

# 3. Jupyter practice

## Running Python code in Jupyter cells

You can run a cell by clicking on it and typing "control-enter" or "command-enter" (Mac).

You can run a cell and move to the next one with "shift-enter".

In [ ]:
# run this cell to find out what the answer is

2 + 3

In [ ]:
# now run this one

2 * 3

In [ ]:
# and now this one

2**3

## Running cells in order

The order in which you run cells matters!

In [ ]:
# step 4
x = x**2

In [ ]:
# step 2
x = x * 2

In [ ]:
# step 1
x = 1

In [ ]:
# step 5
x -= 20

In [ ]:
# step 3
x += 5

If you ran the steps in the right order, the value of `x` should be

```
29
```

In [ ]:
x

Note: you can rearrange the order of the cells by dragging them. Then it's easy to "shift-enter" through them.

It's good practice to set up a notebook so that you can run it from top to bottom. That way, you don't have to remember which cells are "supposed to" be run before others.

## Creating cells

Challenge: create 3 new cells below this one, put some code in them, and run them.

## If you need to start over

If you're not sure which cells have been run, in what order, and need to start over, you can "restart kernel".

<img src="img/restart-kernel.png">

## Last question

Did you run the first cell with all of the "import" statements?

In [ ]:
sqrt

In [ ]:
plt

In [ ]:
data

# 4. What is a "fair" geometry?

## Polsby–Popper score

$$ \mbox{score} = 4\pi \frac{\mbox{area}}{\mbox{perimeter}^2} $$

> It is beyond the scope of this essay, to say nothing of its authors, to say whether a particular district ought to be compact or how compact it ought to be. But we can answer an easier question: whether a compactness criterion complicates the business of gerrymandering. It does. The third criterion will make the gerrymanderer's life a living hell. That's why we're for it.

Polsby, Daniel D.; Popper, Robert D. <a href="https://openyls.law.yale.edu/handle/20.500.13051/17448">"The Third Criterion: Compactness as a procedural safeguard against partisan gerrymandering"</a>. Yale Law & Policy Review. 9 (2): 301–353 (1991).

In [ ]:
#                           x      y
sample_shape = np.array([[ 2.68, 13.54],
                         [-5.8 , 14.15],
                         [-3.45,  7.62],
                         [ 7.99,  9.05],
                         [ 5.74,  1.59],
                         [20.05,  3.23],
                         [21.99,  9.25],
                         [17.49, 13.54],
                         [26.58, 24.26],
                         [12.08, 25.59],
                         [12.59, 17.01],
                         [ 2.88, 19.98],
                         [ 2.68, 13.54]])

In [ ]:
fig, ax = plt.subplots()

#       all x values        all y values
ax.plot(sample_shape[:, 0], sample_shape[:, 1], marker="o")
ax.set_xlabel("x")
ax.set_ylabel("y")

None

### Calculating perimeter

The length of a line segment that spans $\Delta x$ and $\Delta y$ is $\sqrt{\Delta x^2 + \Delta y^2}$.

The perimeter of a polygon is the sum of the lengths of its line segments.

<img src="img/polygon-perimeter.svg" width="500">

Remove the `#` before `result` and replace the `???` with code to calculate the perimeter.

In [ ]:
def perimeter(polygon):
    result = 0

    for i in range(len(polygon) - 1):
        x_this, y_this = polygon[i]
        x_next, y_next = polygon[i + 1]
        # print(x_this, y_this, x_next, y_next)

        # result += ???

    return result

perimeter(sample_shape)

The resulting perimeter should be

```
115.51693885291068
```

### Calculating area

A trapezoid can be broken down into a rectangle and one or two right triangles.

Thea area of a rectangle is $\Delta x \Delta y$.

The area of a right triangle is $\frac{1}{2} \Delta x \Delta y$.

<img src="img/trapezoid-area.svg" width="500">

A polygon can be broken up into trapezoids that all have a rectangular base on the $x$ axis and a triangular top. For each of these trapezoids, we can compute a "signed area," which is positive and equal to its area if $x_1 < x_2$ and equal to its negated area if $x_1 > x_2$. A complete polygon will always have both positive and negative pieces.

The area of the polygon is equal to the sum of signed areas of the trapezoids. Some trapezoids include area outside of the polygon, but it's exactly cancelled out by polygons with negative signed area.

<img src="img/polygon-area-trapezoid-formula.svg" width="500">

Remove the `#` before `trapezoid_signed` and `result` and replace the `???` with code to calculate the area.

In [ ]:
def area(polygon):
    result = 0

    for i in range(len(polygon) - 1):
        x_this, y_this = polygon[i]
        x_next, y_next = polygon[i + 1]
        # print(x_this, y_this, x_next, y_next)

        # trapezoid_signed = ???
        # result += ???

    return result

area(sample_shape)

The resulting area should be

```
374.8152
```

In [ ]:
nearly_circular = np.array([
    [ 2.1 ,  7.02], [ 3.22,  8.94], [ 4.88,  9.79], [ 7.57, 10.41],
    [ 9.44,  9.44], [10.47,  7.95], [10.95,  6.56], [10.54,  4.82],
    [ 9.94,  3.87], [ 8.86,  2.83], [ 7.01,  1.79], [ 4.98,  1.98],
    [ 3.41,  2.64], [ 2.62,  3.82], [ 2.02,  5.32], [ 2.1 ,  7.02]])

In [ ]:
cute_snek = np.array([
    [ 3.95,  9.15], [ 3.59, 10.39], [ 2.47, 10.96], [ 1.44, 10.51],
    [ 0.91,  9.65], [ 2.13,  9.24], [-0.04,  8.98], [ 2.2 ,  9.03],
    [ 1.11,  8.48], [ 1.54,  7.88], [ 2.37,  8.  ], [ 3.21,  7.93],
    [ 4.23,  6.47], [ 4.21,  5.21], [ 4.59,  3.08], [ 5.29,  2.03],
    [ 6.65,  0.6 ], [ 8.56,  0.69], [ 9.8 ,  2.12], [10.57,  4.2 ],
    [10.8 ,  6.64], [11.43,  7.67], [12.48,  7.62], [12.84,  6.14],
    [13.1 ,  3.53], [13.29,  1.62], [13.91,  0.5 ], [15.34,  0.33],
    [16.73,  1.31], [17.73,  2.75], [18.69,  4.8 ], [19.5 ,  6.02],
    [21.46,  7.  ], [22.34,  7.29], [20.65,  7.45], [19.26,  7.02],
    [18.02,  5.99], [17.23,  4.49], [16.2 ,  2.98], [14.72,  2.65],
    [14.1 ,  3.77], [14.25,  5.52], [13.96,  6.95], [13.43,  8.74],
    [12.05,  9.46], [10.26,  9.03], [ 9.54,  7.45], [ 9.18,  5.71],
    [ 8.58,  3.45], [ 8.03,  2.44], [ 6.86,  2.52], [ 6.02,  3.71],
    [ 5.52,  5.19], [ 5.28,  7.06], [ 4.61,  8.11], [ 3.95,  9.15]])

### Test it on more shapes

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

ax1.plot(nearly_circular[:, 0], nearly_circular[:, 1], marker="o")
ax1.set_xlabel("x")
ax1.set_ylabel("y")

ax2.plot(cute_snek[:, 0], cute_snek[:, 1], marker="o")
ax2.set_xlabel("x")
ax2.set_ylabel("y")

None

In [ ]:
def polsby_popper_score(polygon):
    return 4*np.pi * area(polygon) / perimeter(polygon)**2

In [ ]:
polsby_popper_score(nearly_circular)

In [ ]:
polsby_popper_score(cute_snek)

## Analysis of real data

In [ ]:
district_shapes = await data.json_zip("ND-district-shapes.json.zip")

In [ ]:
fig, ax = plt.subplots()

for polygon in district_shapes:
    ax.plot([x for x, y in polygon], [y for x, y in polygon])

ax.set_xlabel("kilometers east")
ax.set_ylabel("kilometers north")

None

Compute the Polsby–Popper score for all the districts in North Dakota.

* How many are below 0.3?
* How many are between 0.3 and 0.7?
* How many are above 0.7?

Don't just count them. Use Python to count for you!

# 5. Pick a state to analyze in more detail

<img src="img/select-a-state.svg" width="800">

In [ ]:
district_gdf = await data.geojson_zip("legislative-district-shapes/ND-upper-house.geojson.zip")

In [ ]:
blocks_gdf = await data.parquet("census-block-shapes/ND-census-blocks.parquet")

In [ ]:
voterfile_df = await data.csv_zip("redistricting-data-hub-voter-files/ND-2022-voter-file.csv.zip")